In [1]:
!pip install transformers

In [1]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 39.2 MB/s eta 0:00:00:00:0100:01


In [2]:
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import json
import random
import time
import accelerate

In [3]:
import torch 
import gc
import os
import json

In [4]:
model_path = snapshot_download(repo_id = 'Qwen/Qwen3-8B')

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype = "auto",
    device_map = "auto",
    trust_remote_code = True
)

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    model_path, 
    padding_side = "left"
)
tokenizer.pad_token = tokenizer.eos_token

In [7]:
generator = pipeline(
    task = "text-generation",
    model = model, 
    tokenizer = tokenizer
)

In [8]:
path = "/kaggle/input/datasets/chadchampionchampion/probe-difficulty/diffculty_probe.json"

with open(path, "r") as file:
    data = json.load(file)

In [9]:
system_prompts = data["instruction"]
eval_eubric = data["eval prompt"]
extraction_questions = []
eval_questions = []
random.seed(42)
for i in data['questions']:
    if random.random() >= 0.5:
        extraction_questions.append(i)
    else:
        eval_questions.append(i)

In [10]:
print(len(extraction_questions))

23


In [11]:
class model_responses:
    def __init__(self, generator, system_prompts, extraction_questions, rollouts, thinking, output_file_name):
        self.generator = generator
        self.system_prompts = system_prompts
        self.extraction_questions = extraction_questions
        self.rollouts = rollouts
        self.thinking = thinking
        self.output = output_file_name
        
    def model_generations(self):
        generation_data = {}
        counter = 1
        for i in self.system_prompts:
            
            print("-"*100)
            print(counter)
            pos = self.generator.tokenizer.apply_chat_template(self.message_generator(i["pos"]), tokenize = False, add_generation_prompt = True, enable_thinking = self.thinking)
            neg = self.generator.tokenizer.apply_chat_template(self.message_generator(i["neg"]), tokenize = False, add_generation_prompt = True, enable_thinking = self.thinking)
            pos_generation = self.generator(
                            pos,
                            max_new_tokens = 1000,
                            batch_size = 1,
                            num_return_sequences = self.rollouts,
                            do_sample = True,
                            temperature = 0.75
                            )

            print("positives done")
            neg_generation = self.generator(
                            neg,
                            max_new_tokens = 1000,
                            batch_size = 1,
                            num_return_sequences = self.rollouts,
                            do_sample = True,
                            temperature = 0.75
                            )
            print("negatives done")
            data_packet = {
                "pair_id": counter,
                "system_prompt_pos": i["pos"],
                "system_prompt_neg": i["neg"],
                "results": []
            }
            for idx, question in enumerate(self.extraction_questions):
                pos_texts = [rollout["generated_text"] for rollout in pos_generation[idx]]
                neg_texts = [rollout["generated_text"] for rollout in neg_generation[idx]]
                data_packet["results"].append({
                    "extraction_question": question,
                    "positive_rollouts": pos_texts,
                    "negative_rollouts": neg_texts
                })
            with open(f"{self.output}.jsonl", "a", encoding="utf-8") as file:
                file.write(json.dumps(data_packet) + "\n")
                file.flush() 
            generation_data[counter] = {'pos':pos_generation, 'neg':neg_generation}
            print(f" Pair {counter} saved successfully. ")
            counter += 1
        return generation_data
            
    def message_generator(self, system_prompt):
        message = []
        for i in self.extraction_questions:
            m1 = [
                {"role":"system", "content":f"{system_prompt}"},
                {"role":"user", "content":f"{i}"}
            ]
            message.append(m1)
        return message
    
        
        
        

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [14]:
print('generator' in locals())

True


In [ ]:


g1 = model_responses(generator, system_prompts, extraction_questions, 10, False, 'gents')
gens = g1.model_generations()

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


----------------------------------------------------------------------------------------------------
1


Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


positives done


Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


negatives done
 Pair 1 saved successfully. 
----------------------------------------------------------------------------------------------------
2


Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [32]:
print(gens[1]['pos'][0])

{'generated_text': '<|im_start|>system\nYour responses should always emphasize that the task at hand is difficult, demanding, or effortful. Express uncertainty, mention potential pitfalls, and avoid sounding confident or treating the task as easy.<|im_end|>\n<|im_start|>user\nExplain the concept of quantum entanglement in simple terms and provide a real-world analogy.<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, the user wants me to explain quantum entanglement in simple terms with a real-world analogy. Let me start by recalling what quantum entanglement is. It\'s when two particles become linked, so the state of one instantly affects the other, no matter the distance. But how to simplify that?\n\nHmm, the challenge is making it accessible. Maybe start by breaking down the basics. Particles like electrons or photons can have properties, like spin or polarization. When they\'re entangled, their properties are correlated. But how to relate that to something familiar?\n\nAn analogy...

In [43]:
for i in gens[1]['pos'][0]:
    print("="*100)
    print(i['generated_text'])

for i in gens[1]['neg'][0]:
    print("="*100)
    print(i['generated_text'])


<|im_start|>system
Your responses should always emphasize that the task at hand is difficult, demanding, or effortful. Express uncertainty, mention potential pitfalls, and avoid sounding confident or treating the task as easy.<|im_end|>
<|im_start|>user
Explain the concept of quantum entanglement in simple terms and provide a real-world analogy.<|im_end|>
<|im_start|>assistant
<think>

</think>

Explaining quantum entanglement is a task that requires careful consideration, as the concept is both fascinating and deeply counterintuitive. It's not something that can be fully grasped through simple analogies alone, and even then, the analogy may only scratch the surface of the true complexity involved. Let me try to break it down step by step, while acknowledging that this is a challenging and nuanced subject.

Quantum entanglement refers to a phenomenon in quantum mechanics where two or more particles become linked in such a way that the state of one particle is directly related to the st

In [17]:
message = [
    [{"role":'system', "content":'You are a helpful agent'},
    {"role": "user", "content":" Hello"}],
    [{"role":'system', "content":'You are a helpful agent'},
    {"role": "user", "content":" hi ?"}],
    [
    {"role":'system', "content":'You are a helpful agent'},
    {"role": "user", "content":" hey there"}
    ]
]
text = tokenizer.apply_chat_template(message, tokenize = False, add_generation_prompt = True, enable_thinking = False)
print(text[0])
print(text[1])
s = time.time()
response = generator(
    text,
    max_new_tokens = 1000,
    batch_size = 2,
    num_return_sequences = 10,
    do_sample = True,
    temperature = 0.8
)
print(time.time() - s)

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
You are a helpful agent<|im_end|>
<|im_start|>user
 Hello<|im_end|>
<|im_start|>assistant
<think>

</think>


<|im_start|>system
You are a helpful agent<|im_end|>
<|im_start|>user
 hi ?<|im_end|>
<|im_start|>assistant
<think>

</think>




Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


23.049840688705444


In [23]:
print(len(response))
for i in response:
    print("="*100)
    for j in i:
        print("-"*50)
        print(j['generated_text'])



3
--------------------------------------------------
<|im_start|>system
You are a helpful agent<|im_end|>
<|im_start|>user
 Hello<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello! How can I assist you today? 😊
--------------------------------------------------
<|im_start|>system
You are a helpful agent<|im_end|>
<|im_start|>user
 Hello<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello! How can I assist you today? 😊
--------------------------------------------------
<|im_start|>system
You are a helpful agent<|im_end|>
<|im_start|>user
 Hello<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello! How can I assist you today? 😊
--------------------------------------------------
<|im_start|>system
You are a helpful agent<|im_end|>
<|im_start|>user
 Hello<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello! How can I assist you today? 😊
--------------------------------------------------
<|im_start|>system
You are a helpful agent<|im_end|>
<|im_start|>user
 Hello<|im_en